# 02. Cortex Analyst ― Semantic View を作り、精度を上げる

**Cortex Analyst** は、自然言語の質問を SQL に変換して実行する機能です。
その「辞書」になるのが **Semantic View** です。このNotebookでは、

1. テーブルを並べただけの **最小構成（v1）** を作り、
2. 説明・同義語・社内 KPI・検証済みクエリを入れた **改善版（v2）** を作り、
3. **同じ 9 問の質問で正答率を自動採点**して、改善の効果を数字で確かめます。

**前提**：`01_explore_data.ipynb` を実行済みであること

## Step 0. セッションの準備

Notebook はそれぞれ別の Python 実行環境で動くため、前の Notebook で作った変数や接続は引き継がれません。
そこで各 Notebook の最初に、Snowflake への接続（`session`）を取得し、ロール・ウェアハウス・スキーマをそろえます。
Python のセルから SQL を実行するには、この `session` が必要です。

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()   # Notebook がすでに張っている Snowflake への接続を取得

DB, SCHEMA = "CORTEX_QUICK_HOL_DB", "CORTEX_QUICK_HOL_SCHEMA"
HOL_ROLE, HOL_WH = "CORTEX_QUICK_HOL_ROLE", "CORTEX_QUICK_HOL_WH"

session.use_role("ACCOUNTADMIN")
session.use_warehouse(HOL_WH)
session.use_schema(f"{DB}.{SCHEMA}")
print("ロール:", session.get_current_role(), "/ ウェアハウス:", session.get_current_warehouse())

## Step 1. Semantic View とは

AI にテーブルだけを渡すと、「売上」「停滞商談」がどのカラムのどんな計算なのかを列名から**推測**します。
推測がぶれると、同じ質問でも答えが変わったり、社内の数字と合わなくなったりします。Semantic View は、その推測をなくすための「データの辞書」です。

| 要素 | 意味 | 例 |
|---|---|---|
| 論理テーブル | 物理テーブルに分かりやすい名前を付けたもの | `TRANSACTIONS` = `SALES_TRANSACTIONS_FACT_TABLE` |
| ディメンション | 切り口（GROUP BY に使う項目） | 地域、業種、商談ステージ |
| ファクト | 行ごとの数値 | 取引金額、滞留日数 |
| メトリクス | 集計の定義（**社内 KPI**） | 停滞商談数 = 未クローズかつ滞留40日超の件数 |
| リレーション | テーブル同士のつなぎ方 | 取引 → 顧客（CUSTOMER_ID） |
| 説明・同義語 | 意味や言い換え | 「勝率」= win rate、「解約リスクは 0〜100」 |
| 検証済みクエリ | よくある質問と正解 SQL のペア | 「停滞している商談は何件？」 |

## Step 2. 最小構成の Semantic View（v1）を作る

まずは **テーブルとカラムを並べただけ** の Semantic View を作ります。
説明・同義語・社内 KPI・検証済みクエリは入れていません。製品は、元教材と同じく取引の SKU でカタログに結合しています。

### 方法 A：GUI（Semantic View Autopilot）で作る

1. Snowsight 左メニュー **「Workspaces」** を開き、**「Add new」→「Semantic View」** を選ぶ（Autopilot のウィザードが開きます）
2. 保存先に `CORTEX_QUICK_HOL_DB` / `CORTEX_QUICK_HOL_SCHEMA`、名前に **`SALES_OPS_SV_BASIC`** を入力し、**Next**
3. **SQL Queries（文脈）は空のまま**にする（v1 は「何も教えていない状態」にしたいため）
4. **Select tables** で次の 5 つを選び、**Next**
   - `SALES_TRANSACTIONS_FACT_TABLE` / `CUSTOMER_DIMENSION_TABLE` / `PRODUCT_CATALOG_TABLE` / `SALES_REP_PERFORMANCE_TABLE` / `SALES_PIPELINE_OPPORTUNITIES`
5. **Select columns** はすべて選び、**サンプル値の追加** と **AI による説明の自動生成** は **オフ**にする
6. **Create and save** → **Save and run**。数分で作成されます
7. 作成後、Semantic Studio で **Logical tables / Relationships** を開き、どんな内容が自動で作られたかを見る

> ⚠️ Autopilot はリレーションや説明を自動で推測するため、GUI 版の v1 は下の SQL 版と完全には一致しません。
> **Step 4 の正答率比較を SQL 版と同じ条件で行うため、デモの後は方法 B のセルを実行して v1 を SQL 版にそろえてください。**

> ℹ️ GUI の手順は Snowflake 公式ドキュメントをもとに書いています。画面のラベルや順番が多少違う場合があります。
> 迷ったら **方法 B（SQL）** のセルを実行すれば、同じ名前・同じ設定で作られます（GUI で作ったものは上書きされます）。

### 方法 B：SQL で作る（`CREATE SEMANTIC VIEW`）

In [ ]:
%%sql -r create_sv_basic
-- v1: テーブルとカラムを並べただけの最小構成（説明・同義語・KPI・検証済みクエリなし）
CREATE OR REPLACE SEMANTIC VIEW CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SV_BASIC
  TABLES (
    TRANSACTIONS AS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE PRIMARY KEY (TRANSACTION_ID),
    CUSTOMERS    AS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CUSTOMER_DIMENSION_TABLE      PRIMARY KEY (CUSTOMER_ID),
    PRODUCTS     AS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_CATALOG_TABLE         PRIMARY KEY (PRODUCT_SKU),
    REPS         AS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_REP_PERFORMANCE_TABLE   PRIMARY KEY (SALES_REP_ID),
    PIPELINE     AS CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES  PRIMARY KEY (OPPORTUNITY_ID)
  )
  RELATIONSHIPS (
    TXN_TO_CUSTOMER AS TRANSACTIONS (CUSTOMER_ID)  REFERENCES CUSTOMERS,
    TXN_TO_PRODUCT  AS TRANSACTIONS (PRODUCT_SKU)  REFERENCES PRODUCTS,
    TXN_TO_REP      AS TRANSACTIONS (SALES_REP_ID) REFERENCES REPS
  )
  FACTS (
    TRANSACTIONS.DEAL_SIZE_USD            AS DEAL_SIZE_USD,
    TRANSACTIONS.DISCOUNT_PERCENTAGE      AS DISCOUNT_PERCENTAGE,
    CUSTOMERS.CHURN_RISK_SCORE            AS CHURN_RISK_SCORE,
    CUSTOMERS.CUSTOMER_LIFETIME_VALUE     AS CUSTOMER_LIFETIME_VALUE,
    PRODUCTS.LIST_PRICE_USD               AS LIST_PRICE_USD,
    REPS.QUOTA_ATTAINMENT_PCT             AS QUOTA_ATTAINMENT_PCT,
    REPS.PIPELINE_COVERAGE_RATIO          AS PIPELINE_COVERAGE_RATIO,
    PIPELINE.ESTIMATED_VALUE              AS ESTIMATED_VALUE,
    PIPELINE.DAYS_IN_CURRENT_STAGE        AS DAYS_IN_CURRENT_STAGE
  )
  DIMENSIONS (
    TRANSACTIONS.REGION                   AS REGION,
    TRANSACTIONS.PRODUCT_SKU              AS PRODUCT_SKU,
    TRANSACTIONS.TRANSACTION_DATE         AS TRANSACTION_DATE,
    CUSTOMERS.COMPANY_NAME                AS COMPANY_NAME,
    CUSTOMERS.INDUSTRY                    AS INDUSTRY,
    CUSTOMERS.COMPANY_SIZE                AS COMPANY_SIZE,
    CUSTOMERS.EXPANSION_POTENTIAL         AS EXPANSION_POTENTIAL,
    PRODUCTS.PRODUCT_TIER                 AS PRODUCT_TIER,
    PRODUCTS.PRODUCT_CATEGORY             AS PRODUCT_CATEGORY,
    REPS.REP_NAME                         AS REP_NAME,
    REPS.TERRITORY                        AS TERRITORY,
    PIPELINE.OPPORTUNITY_ID               AS OPPORTUNITY_ID,
    PIPELINE.CURRENT_STAGE                AS CURRENT_STAGE,
    PIPELINE.COMPETITOR_PRESENT           AS COMPETITOR_PRESENT,
    PIPELINE.DECISION_MAKER_ENGAGED       AS DECISION_MAKER_ENGAGED
  )
  METRICS (
    TRANSACTIONS.TOTAL_DEAL_SIZE AS SUM(TRANSACTIONS.DEAL_SIZE_USD)
  )

### v1 で質問してみる

Cortex Analyst だけを持つ評価用の Agent を作り、質問してみます（Agent については 04 で詳しく扱います。ここでは「Analyst を呼ぶための入れ物」と考えてください）。

In [ ]:
def create_eval_agent(agent_name: str, sv_name: str):
    """指定した Semantic View だけをツールに持つ、評価用の Agent を作る"""
    session.sql(f"""
CREATE OR REPLACE AGENT {DB}.{SCHEMA}.{agent_name}
  COMMENT = 'Evaluation agent (Cortex Analyst only) for {sv_name}'
  FROM SPECIFICATION $$
  models:
    orchestration: auto
  instructions:
    response: Answer briefly with the number. Reply in the same language as the question.
  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: analyst
        description: Sales operations data
  tool_resources:
    analyst:
      semantic_view: {DB}.{SCHEMA}.{sv_name}
      execution_environment:
        type: warehouse
        warehouse: {HOL_WH}
  $$""").collect()
    print("OK:", agent_name, "→", sv_name)

create_eval_agent("EVAL_AGENT_V1", "SALES_OPS_SV_BASIC")

In [ ]:
import json

def ask_agent(question: str, agent: str = "SALES_OPS_AGENT", show_sql: bool = True):
    """Agent に質問し、回答・使ったツール・生成 SQL・検索した文書を表示する"""
    payload = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}]})
    raw = session.sql("SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS R",
                      params=[f"{DB}.{SCHEMA}.{agent}", payload]).collect()[0]["R"]
    resp = json.loads(raw) if isinstance(raw, str) else raw
    texts, sqls, tools, docs = [], [], [], []
    for c in resp.get("content", []):
        if c.get("type") == "text" and c.get("text", "").strip():
            texts.append(c["text"])
        elif c.get("type") == "tool_use":
            tools.append(c.get("tool_use", {}).get("name"))
        elif c.get("type") == "tool_result":
            for x in c.get("tool_result", {}).get("content", []):
                j = x.get("json", {})
                if j.get("sql"):
                    sqls.append(j["sql"])
                for r in j.get("search_results", []) or []:
                    if r.get("doc_title"):
                        docs.append(r["doc_title"])
    print(f"Q: {question}")
    print(f"（使ったツール: {', '.join(t for t in tools if t) or 'なし'}）\n")
    print("\n".join(texts))
    if show_sql:
        for s in sqls:
            print("\n--- Cortex Analyst が生成した SQL ---\n" + s)
    if docs:
        print("\n--- 参照した文書 ---\n" + "\n".join(sorted(set(docs))))
    return resp

In [ ]:
_ = ask_agent("停滞している商談は何件？", agent="EVAL_AGENT_V1")

> 💡 回答の数字と、生成された SQL の条件を見てください。「停滞」の定義（40日超・未クローズ）を知らないため、
> Agent は自分で条件を推測するか、答えられないと返します。01 で確かめた正解は **43 件** です。

## Step 3. 改善版の Semantic View（v2）を作る

v1 に次を加えた改善版を、YAML 形式で作ります（`SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML`）。

| # | 追加するもの | 解決する問題 |
|---|---|---|
| 1 | **説明（description）と同義語（synonyms）**（日本語含む） | 「解約リスクは 0〜100」「勝率＝win rate」などカラム名だけでは伝わらない意味を補う |
| 2 | **リレーションの修正**（取引 → `PRODUCT_PLAN_VIEW`） | v1 では製品の結合が 0 件。プラン単位でつなぎ直す |
| 3 | **社内 KPI のメトリクス** | 停滞商談・失注リスク商談・勝率・推奨カバレッジ未満の人数などを、PDF レポートの基準どおりに定義 |
| 4 | **検証済みクエリ（verified_queries）** | よく聞かれる質問に「お墨付きの SQL」を用意。`use_as_onboarding_question: true` で Snowflake Intelligence の質問候補にも出る |

追加した社内 KPI の例：

| メトリクス | 定義 | 根拠 |
|---|---|---|
| `STALLED_DEAL_COUNT` | 未クローズで現ステージ滞留が **40日超** の件数 | Customer Intelligence レポート |
| `AT_RISK_DEAL_COUNT` / `AT_RISK_PIPELINE_VALUE` | 未クローズ・**競合あり・決裁者未接触** の件数／金額 | 同上 |
| `WIN_RATE_PCT` | 受注 ÷（受注＋失注）× 100 | 社内の定義 |
| `REPS_BELOW_COVERAGE_TARGET` | パイプラインカバレッジ **1.6 倍未満** の営業担当数 | Sales Performance レポート |
| `EXPANSION_TARGET_CUSTOMERS` | 解約リスク **30 未満** かつ拡大余地 High/Medium の顧客数 | 社内の定義 |

### 方法 A：GUI で作る（YAML をアップロード）

v2 は同義語・KPI・検証済みクエリが多く、画面で 1 つずつ入力すると時間がかかります。GUI では **できあがった YAML を読み込む**方法を使います。

1. GitHub の [`semantic_views/sales_ops_semantic_view.yaml`](../semantic_views/sales_ops_semantic_view.yaml) を PC にダウンロードする
   （Git 連携したワークスペースなら、左のファイル一覧の `semantic_views/` にもあります）
2. **「Workspaces」→「Add new」→「Semantic View」** を開き、保存先 `CORTEX_QUICK_HOL_DB` / `CORTEX_QUICK_HOL_SCHEMA` を選ぶ
3. 文脈の指定で **YAML specification のアップロード** を選び、ダウンロードした YAML を指定する
4. 作成されたら Semantic Studio で **Metrics**（`STALLED_DEAL_COUNT` など）と **Verified queries** が入っていることを確認する
5. （デモ）**Synonyms** に同義語を 1 つ追加する、**Verified queries** に質問を 1 つ追加するなど、GUI での編集を試す

> ⚠️ 名前は YAML の中で **`SALES_OPS_SEMANTIC_VIEW`** に決まっています。Step 4 の正答率はこの内容が前提なので、**GUI で大きく変更した場合は方法 B のセルを実行して元に戻してください。**

> ℹ️ GUI の手順は Snowflake 公式ドキュメントをもとに書いています。画面のラベルや順番が多少違う場合があります。
> 迷ったら **方法 B（SQL）** のセルを実行すれば、同じ名前・同じ設定で作られます（GUI で作ったものは上書きされます）。

### 方法 B：SQL で作る（`SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML`）

In [ ]:
%%sql -r create_semantic_view
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA',
  $$
name: SALES_OPS_SEMANTIC_VIEW
description: >
  Semantic view for Sales Operations analytics covering revenue, pipeline health,
  rep performance, customer segmentation, product mix, and marketing attribution.
  営業オペレーション分析用（売上、パイプライン、営業担当の実績、顧客、製品、マーケティング）。

tables:
  - name: TRANSACTIONS
    description: Sales transactions fact table with closed deals, revenue, and attribution. 受注済み取引。
    synonyms: [deals, sales, orders, closed deals, 取引, 受注, 売上明細]
    base_table:
      database: CORTEX_QUICK_HOL_DB
      schema: CORTEX_QUICK_HOL_SCHEMA
      table: SALES_TRANSACTIONS_FACT_TABLE
    primary_key:
      columns: [TRANSACTION_ID]
    dimensions:
      - name: TRANSACTION_ID
        synonyms: [deal id, order id, 取引ID]
        description: Unique transaction identifier
        expr: TRANSACTION_ID
        data_type: TEXT
      - name: REGION
        synonyms: [geo, geography, 地域, リージョン, エリア]
        description: "Geographic region: North America, EMEA, APAC, LATAM"
        expr: REGION
        data_type: TEXT
        sample_values: [North America, EMEA, APAC, LATAM]
      - name: LEAD_SOURCE
        synonyms: [source, acquisition channel, リードソース, 流入経路]
        description: "Lead source: Inbound, Outbound, Event, Partner"
        expr: LEAD_SOURCE
        data_type: TEXT
        sample_values: [Inbound, Outbound, Event, Partner]
      - name: CUSTOMER_ID
        description: Foreign key to customers dimension
        expr: CUSTOMER_ID
        data_type: TEXT
      - name: PRODUCT_SKU
        description: Foreign key to products dimension
        expr: PRODUCT_SKU
        data_type: TEXT
      - name: SALES_REP_ID
        description: Foreign key to sales reps dimension
        expr: SALES_REP_ID
        data_type: TEXT
    time_dimensions:
      - name: TRANSACTION_DATE
        synonyms: [close date, sale date, deal date, 取引日, 受注日]
        description: Date the transaction was recorded
        expr: TRANSACTION_DATE
        data_type: DATE
      - name: FORECAST_CLOSE_DATE
        synonyms: [expected close, forecasted date, 予定クローズ日]
        description: Originally forecasted close date
        expr: FORECAST_CLOSE_DATE
        data_type: DATE
      - name: ACTUAL_CLOSE_DATE
        synonyms: [real close date, 実クローズ日]
        description: Actual date the deal closed
        expr: ACTUAL_CLOSE_DATE
        data_type: DATE
    facts:
      - name: DEAL_SIZE_USD
        synonyms: [deal value, revenue, amount, ARR, 取引金額, 受注金額]
        description: Deal size in USD
        expr: DEAL_SIZE_USD
        data_type: NUMBER
      - name: QUANTITY
        synonyms: [units, qty, 数量]
        description: Quantity of product units
        expr: QUANTITY
        data_type: NUMBER
      - name: DISCOUNT_PERCENTAGE
        synonyms: [discount, discount pct, 値引率, 割引率]
        description: Discount percentage applied
        expr: DISCOUNT_PERCENTAGE
        data_type: NUMBER
      - name: SALES_CYCLE_DAYS
        synonyms: [cycle length, time to close, 商談期間, セールスサイクル]
        description: Days from opportunity creation to close
        expr: SALES_CYCLE_DAYS
        data_type: NUMBER
      - name: CLOSE_PROBABILITY_AT_START
        synonyms: [initial probability, 初期受注確度]
        description: Win probability at deal start
        expr: CLOSE_PROBABILITY_AT_START
        data_type: NUMBER
    metrics:
      - name: TOTAL_REVENUE
        synonyms: [total sales, gross revenue, 売上, 売上合計, 総売上]
        description: Total revenue in USD
        expr: SUM(DEAL_SIZE_USD)
      - name: TOTAL_DEALS
        synonyms: [deal count, number of deals, 取引件数, 受注件数]
        description: Total number of transactions
        expr: COUNT(TRANSACTION_ID)
      - name: AVG_DEAL_SIZE
        synonyms: [average deal, mean deal size, 平均取引金額, 平均単価]
        description: Average deal size in USD
        expr: AVG(DEAL_SIZE_USD)
      - name: AVG_DISCOUNT
        synonyms: [average discount, 平均値引率]
        description: Average discount percentage
        expr: AVG(DISCOUNT_PERCENTAGE)
      - name: AVG_SALES_CYCLE
        synonyms: [average cycle, mean time to close, 平均商談期間]
        description: Average days to close
        expr: AVG(SALES_CYCLE_DAYS)

  - name: CUSTOMERS
    description: Customer dimension with firmographics, LTV, churn risk, expansion potential. 顧客マスタ。
    synonyms: [accounts, clients, companies, 顧客, 取引先, 企業]
    base_table:
      database: CORTEX_QUICK_HOL_DB
      schema: CORTEX_QUICK_HOL_SCHEMA
      table: CUSTOMER_DIMENSION_TABLE
    primary_key:
      columns: [CUSTOMER_ID]
    dimensions:
      - name: CUSTOMER_ID
        synonyms: [account id, client id, 顧客ID]
        description: Unique customer identifier
        expr: CUSTOMER_ID
        data_type: TEXT
      - name: COMPANY_NAME
        synonyms: [account name, company, client name, 会社名, 顧客名]
        description: Customer company name
        expr: COMPANY_NAME
        data_type: TEXT
      - name: INDUSTRY
        synonyms: [vertical, sector, 業種, 業界]
        description: "Industry: Technology, Finance, Healthcare, Manufacturing, Retail"
        expr: INDUSTRY
        data_type: TEXT
        sample_values: [Technology, Finance, Healthcare, Manufacturing, Retail]
      - name: COMPANY_SIZE
        synonyms: [segment, account size, 企業規模, セグメント]
        description: "Company size: SMB, Mid-Market, Enterprise"
        expr: COMPANY_SIZE
        data_type: TEXT
        sample_values: [SMB, Mid-Market, Enterprise]
      - name: ANNUAL_REVENUE_BAND
        synonyms: [revenue band, 年商規模]
        description: Annual revenue band
        expr: ANNUAL_REVENUE_BAND
        data_type: TEXT
      - name: EXPANSION_POTENTIAL
        synonyms: [growth potential, upsell potential, 拡販余地, アップセル余地]
        description: "Expansion potential: High, Medium, Low"
        expr: EXPANSION_POTENTIAL
        data_type: TEXT
        sample_values: [High, Medium, Low]
    time_dimensions:
      - name: FIRST_PURCHASE_DATE
        synonyms: [customer since, acquisition date, 初回購入日]
        description: Date of first purchase
        expr: FIRST_PURCHASE_DATE
        data_type: DATE
    facts:
      - name: CUSTOMER_LIFETIME_VALUE
        synonyms: [CLV, LTV, lifetime value, 顧客生涯価値]
        description: Customer lifetime value in USD
        expr: CUSTOMER_LIFETIME_VALUE
        data_type: NUMBER
      - name: CHURN_RISK_SCORE
        synonyms: [churn risk, attrition risk, 解約リスク, チャーンリスク]
        description: Churn risk score on a 0-100 scale (not 0-1; higher = more likely to churn, below 30 = low risk)
        expr: CHURN_RISK_SCORE
        data_type: NUMBER
      - name: TOTAL_CONTRACTS_COUNT
        synonyms: [contract count, 契約数]
        description: Total contracts with this customer
        expr: TOTAL_CONTRACTS_COUNT
        data_type: NUMBER
    metrics:
      - name: EXPANSION_TARGET_CUSTOMERS
        synonyms: [expansion targets, upsell targets, 拡大対象顧客, アップセル対象]
        description: >
          Customers with low churn risk (score below 30 on the 0-100 scale) and High or Medium expansion potential.
          解約リスク30未満かつ拡大余地 High/Medium の顧客数。
        expr: COUNT_IF(CHURN_RISK_SCORE < 30 AND EXPANSION_POTENTIAL IN ('High', 'Medium'))
      - name: AVG_LIFETIME_VALUE
        synonyms: [average CLV, mean LTV, 平均LTV]
        description: Average customer lifetime value
        expr: AVG(CUSTOMER_LIFETIME_VALUE)
      - name: AVG_CHURN_RISK
        synonyms: [average churn risk, 平均解約リスク]
        description: Average churn risk score
        expr: AVG(CHURN_RISK_SCORE)
      - name: TOTAL_CUSTOMERS
        synonyms: [customer count, 顧客数]
        description: Total unique customers
        expr: COUNT(DISTINCT CUSTOMER_ID)

  - name: PRODUCT_PLANS
    description: >
      Subscription plans (tier x billing frequency) that each sales transaction is sold under.
      Join key PRODUCT_SKU matches SALES_TRANSACTIONS.PRODUCT_SKU (e.g. ENT-ANNUAL).
      取引が紐づく料金プラン（ティア×請求サイクル）。
    synonyms: [plan, product, product tier, subscription plan, プラン, 製品, 料金プラン]
    base_table:
      database: CORTEX_QUICK_HOL_DB
      schema: CORTEX_QUICK_HOL_SCHEMA
      table: PRODUCT_PLAN_VIEW
    primary_key:
      columns: [PRODUCT_SKU]
    dimensions:
      - name: PLAN_SKU
        synonyms: [plan SKU, プランSKU]
        description: Plan SKU used in transactions (e.g. ENT-ANNUAL)
        expr: PRODUCT_SKU
        data_type: TEXT
        sample_values: [ENT-ANNUAL, PRO-MONTHLY, STARTER-ANNUAL]
      - name: PRODUCT_TIER
        synonyms: [tier, pricing tier, plan tier, edition, ティア, プラン, エディション, 製品ティア]
        description: "Product tier: Starter, Professional, Enterprise"
        expr: PRODUCT_TIER
        data_type: TEXT
        sample_values: [Starter, Professional, Enterprise]
      - name: BILLING_FREQUENCY
        synonyms: [billing cycle, payment frequency, 請求サイクル, 支払サイクル]
        description: "Billing frequency of the plan: Annual or Monthly"
        expr: BILLING_FREQUENCY
        data_type: TEXT
        sample_values: [Annual, Monthly]
    facts:
      - name: PLAN_AVG_LIST_PRICE_USD
        synonyms: [plan list price, 平均定価]
        description: Average list price of catalog products in this plan (USD)
        expr: AVG_LIST_PRICE_USD
        data_type: NUMBER
      - name: PLAN_AVG_RENEWAL_RATE_PCT
        synonyms: [plan renewal rate, プラン更新率]
        description: Average renewal rate (percent, 0-100) of catalog products in this plan
        expr: AVG_RENEWAL_RATE_PCT
        data_type: NUMBER
      - name: PLAN_AVG_UPSELL_RATE_PCT
        synonyms: [plan upsell rate, プランのアップセル率]
        description: Average upsell rate (percent, 0-100) of catalog products in this plan
        expr: AVG_UPSELL_RATE_PCT
        data_type: NUMBER
      - name: PLAN_AVG_CONTRACT_LENGTH_MONTHS
        synonyms: [plan contract length, プランの平均契約期間]
        description: Average contract length in months of catalog products in this plan
        expr: AVG_CONTRACT_LENGTH_MONTHS
        data_type: NUMBER

  - name: PRODUCT_CATALOG
    description: Product catalog with pricing, tiers, renewal and upsell rates. 製品カタログ。
    synonyms: [catalog, catalog SKUs, product list, 製品カタログ, 商品一覧]
    base_table:
      database: CORTEX_QUICK_HOL_DB
      schema: CORTEX_QUICK_HOL_SCHEMA
      table: PRODUCT_CATALOG_TABLE
    primary_key:
      columns: [PRODUCT_SKU]
    dimensions:
      - name: PRODUCT_SKU
        synonyms: [catalog SKU, product id, 製品ID]
        description: Unique product SKU
        expr: PRODUCT_SKU
        data_type: TEXT
      - name: PRODUCT_NAME
        synonyms: [name, product, item name, 製品名, 商品名]
        description: Product display name
        expr: PRODUCT_NAME
        data_type: TEXT
      - name: PRODUCT_CATEGORY
        synonyms: [category, product line, 製品カテゴリ]
        description: Product category grouping
        expr: PRODUCT_CATEGORY
        data_type: TEXT
      - name: PRODUCT_FEATURE
        synonyms: [feature, capability, 機能]
        description: Primary product feature
        expr: PRODUCT_FEATURE
        data_type: TEXT
      - name: CATALOG_PRODUCT_TIER
        synonyms: [catalog tier, カタログのティア]
        description: "Product tier: Starter, Professional, Enterprise"
        expr: PRODUCT_TIER
        data_type: TEXT
        sample_values: [Starter, Professional, Enterprise]
      - name: CATALOG_BILLING_FREQUENCY
        synonyms: [billing cycle, payment frequency, 請求サイクル]
        description: "Billing frequency: Monthly, Annual, Multi-year"
        expr: BILLING_FREQUENCY
        data_type: TEXT
        sample_values: [Monthly, Annual, Multi-year]
      - name: TYPICAL_DISCOUNT_RANGE
        synonyms: [discount range, 標準値引幅]
        description: Typical discount range
        expr: TYPICAL_DISCOUNT_RANGE
        data_type: TEXT
    facts:
      - name: LIST_PRICE_USD
        synonyms: [list price, MSRP, 定価]
        description: List price in USD
        expr: LIST_PRICE_USD
        data_type: NUMBER
      - name: AVERAGE_CONTRACT_LENGTH_MONTHS
        synonyms: [contract length, avg term, 平均契約期間]
        description: Average contract length in months
        expr: AVERAGE_CONTRACT_LENGTH_MONTHS
        data_type: NUMBER
      - name: RENEWAL_RATE_PCT
        synonyms: [renewal rate, retention rate, 更新率]
        description: Product renewal rate percentage
        expr: RENEWAL_RATE_PCT
        data_type: NUMBER
      - name: UPSELL_RATE_PCT
        synonyms: [upsell rate, expansion rate, アップセル率]
        description: Product upsell rate percentage
        expr: UPSELL_RATE_PCT
        data_type: NUMBER
    metrics:
      - name: AVG_LIST_PRICE
        synonyms: [average price, 平均定価]
        description: Average list price in USD
        expr: AVG(LIST_PRICE_USD)
      - name: AVG_RENEWAL_RATE
        synonyms: [average renewal, 平均更新率]
        description: Average renewal rate
        expr: AVG(RENEWAL_RATE_PCT)

  - name: REPS
    description: Sales rep performance with quota, attainment, win rate, pipeline coverage. 営業担当の実績。
    synonyms: [sales reps, salespeople, account executives, AEs, 営業担当, 営業, 担当者]
    base_table:
      database: CORTEX_QUICK_HOL_DB
      schema: CORTEX_QUICK_HOL_SCHEMA
      table: SALES_REP_PERFORMANCE_TABLE
    primary_key:
      columns: [SALES_REP_ID]
    dimensions:
      - name: SALES_REP_ID
        synonyms: [rep id, AE id, 担当者ID]
        description: Unique sales rep identifier
        expr: SALES_REP_ID
        data_type: TEXT
      - name: REP_NAME
        synonyms: [name, sales rep name, AE name, 担当者名, 営業担当名]
        description: Full name of the sales rep
        expr: REP_NAME
        data_type: TEXT
      - name: TERRITORY
        synonyms: [assigned territory, sales territory, 担当エリア, テリトリー]
        description: Sales territory assignment
        expr: TERRITORY
        data_type: TEXT
    time_dimensions:
      - name: HIRE_DATE
        synonyms: [start date, joined date, 入社日]
        description: Date the rep was hired
        expr: HIRE_DATE
        data_type: DATE
    facts:
      - name: QUOTA_USD
        synonyms: [quota, target, sales target, ノルマ, 目標, クォータ]
        description: Annual quota in USD
        expr: QUOTA_USD
        data_type: NUMBER
      - name: QUOTA_ATTAINMENT_PCT
        synonyms: [attainment, quota attainment, 目標達成率, 達成率]
        description: Quota attainment percentage
        expr: QUOTA_ATTAINMENT_PCT
        data_type: NUMBER
      - name: AVG_DEAL_SIZE
        synonyms: [average deal, 担当者別平均取引金額]
        description: Average deal size in USD
        expr: AVG_DEAL_SIZE
        data_type: NUMBER
      - name: WIN_RATE_PCT
        synonyms: [win rate, close rate, 受注率, 勝率]
        description: Win rate percentage
        expr: WIN_RATE_PCT
        data_type: NUMBER
      - name: AVG_SALES_CYCLE_DAYS
        synonyms: [avg cycle, 担当者別平均商談期間]
        description: Average days to close
        expr: AVG_SALES_CYCLE_DAYS
        data_type: NUMBER
      - name: PIPELINE_COVERAGE_RATIO
        synonyms: [coverage ratio, pipeline multiple, パイプラインカバレッジ]
        description: Pipeline coverage ratio (recommended minimum 1.6x)
        expr: PIPELINE_COVERAGE_RATIO
        data_type: NUMBER
      - name: ACTIVE_OPPORTUNITIES_COUNT
        synonyms: [active opps, open opportunities, 進行中商談数]
        description: Active opportunity count
        expr: ACTIVE_OPPORTUNITIES_COUNT
        data_type: NUMBER
    metrics:
      - name: REPS_ABOVE_QUOTA
        synonyms: [reps over quota, reps hitting quota, 目標達成者数, 目標を達成した営業担当の数]
        description: Number of sales reps whose quota attainment is above 100 percent
        expr: COUNT_IF(QUOTA_ATTAINMENT_PCT > 100)
      - name: REPS_BELOW_COVERAGE_TARGET
        synonyms: [coverage gap reps, カバレッジ不足の営業担当, 推奨カバレッジ未満]
        description: >
          Number of reps below the recommended minimum pipeline coverage of 1.6x
          (company target from the Sales Performance Analysis report). 推奨カバレッジ1.6倍未満の営業担当数。
        expr: COUNT_IF(PIPELINE_COVERAGE_RATIO < 1.6)
      - name: AVG_QUOTA_ATTAINMENT
        synonyms: [average attainment, team attainment, 平均達成率]
        description: Average quota attainment across reps
        expr: AVG(QUOTA_ATTAINMENT_PCT)
      - name: AVG_WIN_RATE
        synonyms: [average win rate, team win rate, 平均受注率]
        description: Average win rate across reps
        expr: AVG(WIN_RATE_PCT)

  - name: PIPELINE
    description: Active pipeline opportunities with stages, probabilities, and engagement signals. 商談パイプライン。
    synonyms: [opportunities, opps, deals in progress, funnel, 商談, パイプライン, 案件]
    base_table:
      database: CORTEX_QUICK_HOL_DB
      schema: CORTEX_QUICK_HOL_SCHEMA
      table: SALES_PIPELINE_OPPORTUNITIES
    primary_key:
      columns: [OPPORTUNITY_ID]
    dimensions:
      - name: OPPORTUNITY_ID
        synonyms: [opp id, 商談ID]
        description: Unique opportunity identifier
        expr: OPPORTUNITY_ID
        data_type: TEXT
      - name: CURRENT_STAGE
        synonyms: [stage, deal stage, pipeline stage, ステージ, フェーズ]
        description: "Stage: Prospecting, Qualification, Proposal, Negotiation, Closed Won, Closed Lost"
        expr: CURRENT_STAGE
        data_type: TEXT
        sample_values: [Prospecting, Qualification, Proposal, Negotiation, Closed Won, Closed Lost]
      - name: COMPETITOR_PRESENT
        synonyms: [competitive deal, has competitor, 競合あり]
        description: Whether a competitor is present
        expr: COMPETITOR_PRESENT
        data_type: BOOLEAN
      - name: DECISION_MAKER_ENGAGED
        synonyms: [DM engaged, 決裁者接触]
        description: Whether decision maker is engaged
        expr: DECISION_MAKER_ENGAGED
        data_type: BOOLEAN
    time_dimensions:
      - name: CREATED_DATE
        synonyms: [opp created, date created, 商談作成日]
        description: Opportunity creation date
        expr: CREATED_DATE
        data_type: DATE
      - name: EXPECTED_CLOSE_DATE
        synonyms: [expected close, target close date, クローズ予定日]
        description: Expected close date
        expr: EXPECTED_CLOSE_DATE
        data_type: DATE
      - name: STAGE_ENTRY_DATE
        synonyms: [entered stage, ステージ移行日]
        description: Date entered current stage
        expr: STAGE_ENTRY_DATE
        data_type: DATE
      - name: LAST_ACTIVITY_DATE
        synonyms: [last touch, last contact, 最終活動日]
        description: Most recent activity date
        expr: LAST_ACTIVITY_DATE
        data_type: DATE
    facts:
      - name: ESTIMATED_VALUE
        synonyms: [opp value, opportunity amount, 商談金額, 見込金額]
        description: Estimated deal value in USD
        expr: ESTIMATED_VALUE
        data_type: NUMBER
      - name: WEIGHTED_FORECAST_VALUE
        synonyms: [weighted value, forecast value, 加重見込金額]
        description: Probability-weighted forecast in USD
        expr: WEIGHTED_FORECAST_VALUE
        data_type: NUMBER
      - name: CLOSE_PROBABILITY
        synonyms: [win probability, 受注確度]
        description: Close probability (0-100)
        expr: CLOSE_PROBABILITY
        data_type: NUMBER
      - name: DAYS_IN_CURRENT_STAGE
        synonyms: [days in stage, stage duration, ステージ滞留日数]
        description: Days in current stage (more than 40 days on an open deal = stalled)
        expr: DAYS_IN_CURRENT_STAGE
        data_type: NUMBER
      - name: ACTIVITY_COUNT_LAST_30_DAYS
        synonyms: [recent activity, activity count, 直近30日の活動数]
        description: Activities in last 30 days
        expr: ACTIVITY_COUNT_LAST_30_DAYS
        data_type: NUMBER
    metrics:
      - name: TOTAL_PIPELINE_VALUE
        synonyms: [pipeline value, パイプライン金額, 商談総額]
        description: Total estimated pipeline value
        expr: SUM(ESTIMATED_VALUE)
      - name: TOTAL_WEIGHTED_FORECAST
        synonyms: [weighted pipeline, total forecast, 加重パイプライン, フォーキャスト]
        description: Total weighted forecast value
        expr: SUM(WEIGHTED_FORECAST_VALUE)
      - name: TOTAL_OPPORTUNITIES
        synonyms: [opp count, 商談数]
        description: Total opportunities
        expr: COUNT(OPPORTUNITY_ID)
      - name: AVG_DAYS_IN_STAGE
        synonyms: [average stage duration, 平均滞留日数]
        description: Average days in current stage
        expr: AVG(DAYS_IN_CURRENT_STAGE)
      - name: OPEN_PIPELINE_VALUE
        synonyms: [open pipeline, active pipeline value, 進行中パイプライン, オープン商談金額]
        description: Total estimated value of open opportunities (excluding Closed Won / Closed Lost)
        expr: SUM(IFF(CURRENT_STAGE NOT IN ('Closed Won', 'Closed Lost'), ESTIMATED_VALUE, 0))
      - name: STALLED_DEAL_COUNT
        synonyms: [stalled deals, stuck deals, 停滞商談, 停滞している商談, 滞留商談]
        description: >
          Company definition of a stalled deal: an open opportunity that has been in its current stage
          for more than 40 days. 社内定義：未クローズで現ステージ滞留が40日超の商談。
        expr: COUNT_IF(DAYS_IN_CURRENT_STAGE > 40 AND CURRENT_STAGE NOT IN ('Closed Won', 'Closed Lost'))
      - name: AT_RISK_DEAL_COUNT
        synonyms: [at-risk deals, deals at risk, 失注リスク商談, リスク商談]
        description: >
          Company definition of an at-risk deal: an open opportunity with a competitor present and
          no decision maker engaged. 社内定義：未クローズ・競合あり・決裁者未接触の商談。
        expr: COUNT_IF(COMPETITOR_PRESENT AND NOT DECISION_MAKER_ENGAGED AND CURRENT_STAGE NOT IN ('Closed Won', 'Closed Lost'))
      - name: AT_RISK_PIPELINE_VALUE
        synonyms: [at-risk pipeline, 失注リスク金額, リスク商談の金額]
        description: Total estimated value of at-risk deals (open, competitor present, no decision maker engaged)
        expr: SUM(IFF(COMPETITOR_PRESENT AND NOT DECISION_MAKER_ENGAGED AND CURRENT_STAGE NOT IN ('Closed Won', 'Closed Lost'), ESTIMATED_VALUE, 0))
      - name: WIN_RATE_PCT
        synonyms: [win rate, close rate, 勝率, 受注率]
        description: >
          Company definition of win rate: Closed Won / (Closed Won + Closed Lost) x 100, using pipeline stages.
          社内定義：受注 ÷（受注＋失注）× 100。
        expr: COUNT_IF(CURRENT_STAGE = 'Closed Won') * 100.0 / NULLIF(COUNT_IF(CURRENT_STAGE IN ('Closed Won', 'Closed Lost')), 0)

  - name: MARKETING
    description: Marketing lead attribution with campaigns, lead scores, and spend. マーケティング施策とリード。
    synonyms: [leads, campaigns, marketing leads, attribution, リード, キャンペーン, マーケティング]
    base_table:
      database: CORTEX_QUICK_HOL_DB
      schema: CORTEX_QUICK_HOL_SCHEMA
      table: MARKETING_ATTRIBUTION_TABLE
    primary_key:
      columns: [LEAD_ID]
    dimensions:
      - name: LEAD_ID
        synonyms: [lead, marketing lead, リードID]
        description: Unique lead identifier
        expr: LEAD_ID
        data_type: TEXT
      - name: LEAD_SOURCE_DETAIL
        synonyms: [source detail, lead channel, リード獲得チャネル]
        description: "Lead source: Google Ads, LinkedIn, Trade Show, Webinar"
        expr: LEAD_SOURCE_DETAIL
        data_type: TEXT
        sample_values: [Google Ads, LinkedIn, Trade Show, Webinar]
      - name: CAMPAIGN_NAME
        synonyms: [campaign, marketing campaign, キャンペーン名]
        description: Marketing campaign name
        expr: CAMPAIGN_NAME
        data_type: TEXT
      - name: FIRST_TOUCH_CHANNEL
        synonyms: [first touch, initial channel, ファーストタッチ]
        description: First marketing touch channel
        expr: FIRST_TOUCH_CHANNEL
        data_type: TEXT
      - name: LAST_TOUCH_CHANNEL
        synonyms: [last touch, final channel, ラストタッチ]
        description: Last marketing touch before conversion
        expr: LAST_TOUCH_CHANNEL
        data_type: TEXT
    facts:
      - name: LEAD_SCORE
        synonyms: [score, lead quality, MQL score, リードスコア]
        description: Lead quality score (0-100)
        expr: LEAD_SCORE
        data_type: NUMBER
      - name: LEAD_TO_OPPORTUNITY_DAYS
        synonyms: [lead conversion time, days to opportunity, 商談化日数]
        description: Days from lead to opportunity
        expr: LEAD_TO_OPPORTUNITY_DAYS
        data_type: NUMBER
      - name: MARKETING_SPEND_ALLOCATED
        synonyms: [spend, marketing cost, ad spend, マーケティング費用, 広告費]
        description: Marketing spend allocated in USD
        expr: MARKETING_SPEND_ALLOCATED
        data_type: NUMBER
    metrics:
      - name: TOTAL_MARKETING_SPEND
        synonyms: [total spend, total ad spend, マーケティング費用合計]
        description: Total marketing spend in USD
        expr: SUM(MARKETING_SPEND_ALLOCATED)
      - name: AVG_LEAD_SCORE
        synonyms: [average lead score, 平均リードスコア]
        description: Average lead quality score
        expr: AVG(LEAD_SCORE)
      - name: TOTAL_LEADS
        synonyms: [lead count, リード数]
        description: Total marketing leads
        expr: COUNT(LEAD_ID)
      - name: AVG_LEAD_CONVERSION_TIME
        synonyms: [average conversion time, 平均商談化日数]
        description: Average days from lead to opportunity
        expr: AVG(LEAD_TO_OPPORTUNITY_DAYS)

relationships:
  - name: TRANSACTIONS_TO_CUSTOMERS
    left_table: TRANSACTIONS
    right_table: CUSTOMERS
    join_type: left_outer
    relationship_type: many_to_one
    relationship_columns:
      - left_column: CUSTOMER_ID
        right_column: CUSTOMER_ID
  - name: TRANSACTIONS_TO_PRODUCTS
    left_table: TRANSACTIONS
    right_table: PRODUCT_PLANS
    join_type: left_outer
    relationship_type: many_to_one
    relationship_columns:
      - left_column: PRODUCT_SKU
        right_column: PRODUCT_SKU
  - name: TRANSACTIONS_TO_REPS
    left_table: TRANSACTIONS
    right_table: REPS
    join_type: left_outer
    relationship_type: many_to_one
    relationship_columns:
      - left_column: SALES_REP_ID
        right_column: SALES_REP_ID

verified_queries:
  - name: stalled_deals_jp
    question: 停滞している商談は何件？
    use_as_onboarding_question: true
    sql: |
      SELECT COUNT(*) AS STALLED_DEALS
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES
      WHERE DAYS_IN_CURRENT_STAGE > 40 AND CURRENT_STAGE NOT IN ('Closed Won', 'Closed Lost')
  - name: win_rate_jp
    question: 商談の勝率は？
    use_as_onboarding_question: true
    sql: |
      SELECT COUNT_IF(CURRENT_STAGE = 'Closed Won') * 100.0
             / NULLIF(COUNT_IF(CURRENT_STAGE IN ('Closed Won', 'Closed Lost')), 0) AS WIN_RATE_PCT
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES
  - name: revenue_by_tier_jp
    question: 製品ティア別の売上を教えて
    use_as_onboarding_question: true
    sql: |
      SELECT pr.PRODUCT_TIER, SUM(t.DEAL_SIZE_USD) AS TOTAL_REVENUE
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t
      JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_PLAN_VIEW pr ON t.PRODUCT_SKU = pr.PRODUCT_SKU
      GROUP BY pr.PRODUCT_TIER ORDER BY TOTAL_REVENUE DESC
  - name: revenue_by_region
    question: What is the total revenue by region?
    sql: |
      SELECT t.REGION, SUM(t.DEAL_SIZE_USD) AS TOTAL_REVENUE
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t
      GROUP BY t.REGION ORDER BY TOTAL_REVENUE DESC
  - name: top_reps_by_attainment
    question: Which sales reps have the highest quota attainment?
    sql: |
      SELECT r.REP_NAME, r.TERRITORY, r.QUOTA_ATTAINMENT_PCT, r.QUOTA_USD
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_REP_PERFORMANCE_TABLE r
      ORDER BY r.QUOTA_ATTAINMENT_PCT DESC LIMIT 10
  - name: pipeline_by_stage
    question: What is the pipeline value broken down by stage?
    sql: |
      SELECT p.CURRENT_STAGE, COUNT(p.OPPORTUNITY_ID) AS NUM_OPPORTUNITIES,
        SUM(p.ESTIMATED_VALUE) AS TOTAL_VALUE,
        SUM(p.WEIGHTED_FORECAST_VALUE) AS WEIGHTED_VALUE
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES p
      GROUP BY p.CURRENT_STAGE ORDER BY TOTAL_VALUE DESC
  - name: revenue_by_product_tier
    question: How does revenue break down by product tier?
    sql: |
      SELECT pr.PRODUCT_TIER, COUNT(t.TRANSACTION_ID) AS NUM_DEALS,
        SUM(t.DEAL_SIZE_USD) AS TOTAL_REVENUE,
        AVG(t.DEAL_SIZE_USD) AS AVG_DEAL_SIZE
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t
      JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_PLAN_VIEW pr ON t.PRODUCT_SKU = pr.PRODUCT_SKU
      GROUP BY pr.PRODUCT_TIER ORDER BY TOTAL_REVENUE DESC
  - name: top_customers_by_ltv
    question: Who are the top 10 customers by lifetime value?
    sql: |
      SELECT c.COMPANY_NAME, c.INDUSTRY, c.COMPANY_SIZE,
        c.CUSTOMER_LIFETIME_VALUE, c.CHURN_RISK_SCORE, c.EXPANSION_POTENTIAL
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CUSTOMER_DIMENSION_TABLE c
      ORDER BY c.CUSTOMER_LIFETIME_VALUE DESC LIMIT 10
  - name: marketing_spend_by_source
    question: What is the total marketing spend by lead source?
    sql: |
      SELECT m.LEAD_SOURCE_DETAIL, COUNT(m.LEAD_ID) AS NUM_LEADS,
        SUM(m.MARKETING_SPEND_ALLOCATED) AS TOTAL_SPEND,
        AVG(m.LEAD_SCORE) AS AVG_LEAD_SCORE
      FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.MARKETING_ATTRIBUTION_TABLE m
      GROUP BY m.LEAD_SOURCE_DETAIL ORDER BY TOTAL_SPEND DESC
$$,
  FALSE
)

作成できたか確認し、ハンズオン用ロールに参照権限を付けます。

In [ ]:
%%sql -r show_semantic_views
SHOW SEMANTIC VIEWS IN SCHEMA CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA

In [ ]:
for sv in ["SALES_OPS_SV_BASIC", "SALES_OPS_SEMANTIC_VIEW"]:
    session.sql(f"GRANT SELECT ON SEMANTIC VIEW {DB}.{SCHEMA}.{sv} TO ROLE {HOL_ROLE}").collect()
print("OK: Semantic View への SELECT を付与しました")

### SQL で直接クエリしてみる

`SEMANTIC_VIEW()` 関数を使うと、**メトリクス名を指定するだけ**で、定義どおりの集計を SQL から使えます。
社内 KPI が Semantic View の中に「部品」として入ったことが分かります。

In [ ]:
%%sql -r sv_kpis
SELECT * FROM SEMANTIC_VIEW(
  CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SEMANTIC_VIEW
  METRICS PIPELINE.STALLED_DEAL_COUNT, PIPELINE.AT_RISK_DEAL_COUNT, PIPELINE.WIN_RATE_PCT,
          REPS.REPS_BELOW_COVERAGE_TARGET
)

In [ ]:
%%sql -r sv_revenue_by_plan
SELECT * FROM SEMANTIC_VIEW(
  CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_SEMANTIC_VIEW
  DIMENSIONS PRODUCT_PLANS.PRODUCT_TIER
  METRICS TRANSACTIONS.TOTAL_REVENUE
)
ORDER BY TOTAL_REVENUE DESC

### Snowsight で中身を見る（GUI）

1. Snowsight 左メニュー **「AI と ML」→「Cortex Analyst」**（または **「Workspaces」の Semantic Studio**）を開く
2. `CORTEX_QUICK_HOL_DB` の `SALES_OPS_SEMANTIC_VIEW` をクリック
3. **Logical tables / Relationships / Metrics / Verified queries** を開き、`SALES_OPS_SV_BASIC` と見比べる

> Semantic View の編集画面から、同義語や検証済みクエリを GUI で追加することもできます。

## Step 4. 正答率を比べる（v1 vs v2）

同じ 9 問を v1・v2 それぞれに投げ、**Analyst が生成した SQL を実行して、結果に正解の値が含まれるか**で自動採点します。
正解は 01 で SQL を使って確かめた値です。

| # | 質問 | 正解 | ポイント |
|---|---|---|---|
| 1 | 北米の売上合計は？ | $5,947,306.61 | 基本の集計 |
| 2 | Enterprise ティアの売上合計は？ | $7,816,021.78 | 製品の結合（データの落とし穴） |
| 3 | 売上が最も大きい業種はどこ？ | Technology | テーブルをまたぐ集計 |
| 4 | 停滞している商談は何件？ | 43 件 | 社内 KPI |
| 5 | 目標を達成している営業担当は何人？ | 87 人 | 基本の集計 |
| 6 | 解約リスクが低く、拡大余地が High の顧客は何社？ | 25 社 | 「低い」＝30 未満（0〜100 の尺度） |
| 7 | 失注リスクのある商談の金額合計は？ | $3,629,041.99 | 社内 KPI |
| 8 | パイプラインカバレッジが推奨水準を下回る営業担当は何人？ | 63 人 | PDF に書かれた基準（1.6 倍） |
| 9 | 商談の勝率は？ | 69.7% | 社内 KPI |

> 9 問を 5 並列で投げるので、1 回の採点は 30〜60 秒ほどです。AI の回答は毎回少しずつ変わるため、v1 の点数は実行ごとに前後します。

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

EVAL_SET = [
    ("北米の売上合計は？",                                     5947306.61),
    ("Enterprise ティアの売上合計は？",                         7816021.78),
    ("売上が最も大きい業種はどこ？",                            "Technology"),
    ("停滞している商談は何件？",                                43),
    ("目標を達成している営業担当は何人？",                      87),
    ("解約リスクが低く、拡大余地が High の顧客は何社？",        25),
    ("失注リスクのある商談の金額合計は？",                      3629041.99),
    ("パイプラインカバレッジが推奨水準を下回る営業担当は何人？", 63),
    ("商談の勝率は？",                                          69.7),
]

def _matches(expected, values):
    for v in values:
        if v is None:
            continue
        if isinstance(expected, str):
            if expected.lower() in str(v).lower():
                return True
        else:
            try:
                if abs(float(v) - expected) <= max(0.5, abs(expected) * 0.005):
                    return True
            except (TypeError, ValueError):
                pass
    return False

def _grade(agent_fqn, question, expected):
    t = time.time()
    payload = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}]})
    try:
        raw = session.sql("SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS R", params=[agent_fqn, payload]).collect()[0]["R"]
        resp = json.loads(raw) if isinstance(raw, str) else raw
        sqls = [x["json"]["sql"] for c in resp.get("content", []) if c.get("type") == "tool_result"
                for x in c.get("tool_result", {}).get("content", []) if x.get("json", {}).get("sql")]
        values = [v for row in session.sql(sqls[-1]).collect()[:50] for v in row] if sqls else []
        return {"QUESTION": question, "OK": _matches(expected, values), "SEC": round(time.time() - t)}
    except Exception as e:
        return {"QUESTION": question, "OK": False, "SEC": round(time.time() - t), "ERROR": str(e)[:100]}

def evaluate(agent_name: str):
    fqn = f"{DB}.{SCHEMA}.{agent_name}"
    with ThreadPoolExecutor(max_workers=5) as ex:
        rows = list(ex.map(lambda qa: _grade(fqn, *qa), EVAL_SET))
    print(f"=== {agent_name}: {sum(r['OK'] for r in rows)} / {len(rows)} 正解")
    for r in rows:
        print(f"  {'○' if r['OK'] else '×'}  {r['QUESTION']}")
    return rows

In [ ]:
create_eval_agent("EVAL_AGENT_V2", "SALES_OPS_SEMANTIC_VIEW")

result_v1 = evaluate("EVAL_AGENT_V1")
result_v2 = evaluate("EVAL_AGENT_V2")

### 結果を並べる

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "質問": [r["QUESTION"] for r in result_v1],
    "v1（最小構成）": ["○" if r["OK"] else "×" for r in result_v1],
    "v2（改善版）": ["○" if r["OK"] else "×" for r in result_v2],
})
print(f"正答率  v1: {sum(r['OK'] for r in result_v1)}/{len(result_v1)}  →  v2: {sum(r['OK'] for r in result_v2)}/{len(result_v2)}")
df

> ✅ 検証したアカウント（2026年10月）では **v1: 4/9 → v2: 9/9** でした。
>
> v1 が間違えた質問を見ると、**「データのつながり（製品）」「社内用語（停滞・失注リスク・勝率）」「社外の基準（推奨カバレッジ）」** の 3 種類に分かれます。
> どれもモデルを賢くするのではなく、**Semantic View にビジネスの定義を書く**ことで解決しています。
>
> 💡 **自分でも試してみましょう**：v2 で間違える質問を見つけたら、同義語・説明・検証済みクエリを足して `create_semantic_view` を再実行し、もう一度採点します。
> この「評価セットで測る → 定義を足す → もう一度測る」の繰り返しが、本番運用での精度向上の基本です。

### 後片付け（評価用 Agent）

評価用の Agent は以降使わないので削除します（Semantic View v1 は比較用に残しておきます）。

In [ ]:
for a in ["EVAL_AGENT_V1", "EVAL_AGENT_V2"]:
    session.sql(f"DROP AGENT IF EXISTS {DB}.{SCHEMA}.{a}").collect()
print("OK: 評価用 Agent を削除しました")

✅ 02 はここまでです。次は `03_cortex_search.ipynb` に進みます。